# Step 1: Load Data and Clean the LendingClub Dataset

This notebook loads the raw LendingClub files, performs first-pass data cleaning, and saves a cleaned dataset for the modeling notebooks. The cleaning choices below are intentionally documented so later steps can cite exactly what was removed and why.


## 1. Setup

Import the libraries needed for loading and cleaning. Modeling imports are kept out of this notebook so Step 1 stays focused on data preparation.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)


## 2. File paths

The notebook supports both Google Colab and a local project folder. If the data is stored somewhere else, update `DATA_ROOT` below before running the load cell.


In [ ]:
# Prefer the original Google Drive location when running in Colab; otherwise use ./Data.
DATA_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending/Data")
RAW_DATA_DIR = DATA_ROOT / "OriginalData"
MODIFIED_DATA_DIR = DATA_ROOT / "ModifiedData"
MODIFIED_DATA_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = RAW_DATA_DIR / "LCDataDictionary.csv"
RAW_LOANS_PATH = RAW_DATA_DIR / "lending_club_loans.csv"
CLEAN_DATA_PATH = MODIFIED_DATA_DIR / "CleanData.xlsx"
SAMPLE_PATH = MODIFIED_DATA_DIR / "CleanData_sample.xlsx"

print(f"Data root: {DATA_ROOT}")
print(f"Raw loans path: {RAW_LOANS_PATH}")
print(f"Clean output path: {CLEAN_DATA_PATH}")


## 3. Load raw files

`MetaData` contains the LendingClub data dictionary. `Data` contains the raw loan records. The loan file is read with `skiprows=1` because the downloaded CSV includes one non-header line before the real table header.


In [ ]:
required_files = [METADATA_PATH, RAW_LOANS_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required raw data files. Update DATA_ROOT or place files here:\n"
        + "\n".join(missing_files)
    )

MetaData = pd.read_csv(METADATA_PATH, low_memory=False)
Data = pd.read_csv(RAW_LOANS_PATH, low_memory=False, skiprows=1)

print(f"Metadata shape: {MetaData.shape}")
print(f"Raw loan data shape: {Data.shape}")
display(Data.head(3))


## 4. Initial structural cleaning

These steps remove columns or rows that cannot contribute useful modeling signal: completely empty columns, footer rows without valid IDs, and columns with only one observed value.


In [ ]:
# Remove columns that are entirely missing.
all_missing_cols = Data.columns[Data.isna().all()].tolist()
print(f"Columns before: {Data.shape[1]}")
print(f"All-missing columns removed: {len(all_missing_cols)}")

Data = Data.drop(columns=all_missing_cols)
print(f"Columns after: {Data.shape[1]}")


In [ ]:
# Keep only rows with a valid numeric application ID.
rows_before = len(Data)
Data["id"] = pd.to_numeric(Data["id"], errors="coerce")
Data = Data.dropna(subset=["id"]).copy()
Data["id"] = Data["id"].astype(int)

print(f"Rows before ID cleaning: {rows_before}")
print(f"Rows after ID cleaning: {len(Data)}")


In [ ]:
# Remove features that have the same value for every remaining row.
constant_cols = [col for col in Data.columns if Data[col].nunique(dropna=False) <= 1]
print(f"Constant columns removed: {len(constant_cols)}")
display(pd.Series(constant_cols, name="constant_columns"))

Data = Data.drop(columns=constant_cols)
print(f"Data shape after constant-column removal: {Data.shape}")


## 5. Outcome definition

The target variable is `loan_status`. For binary default prediction, keep only finalized loans: `Fully Paid` and `Charged Off`. In-progress or rare statuses are removed because their final outcome is unknown or too sparse.


In [ ]:
display(Data["loan_status"].value_counts(dropna=False))

accepted_outcomes = ["Fully Paid", "Charged Off"]
rows_before = len(Data)
Data = Data[Data["loan_status"].isin(accepted_outcomes)].copy()

print(f"Rows before outcome filtering: {rows_before}")
print(f"Rows after outcome filtering: {len(Data)}")
display(Data["loan_status"].value_counts())


## 6. Small value recoding and missing values

Combine extremely rare home ownership labels and fill delinquency history missing values with 0, since missing in `delinq_2yrs` indicates no recorded delinquencies in this dataset.


In [ ]:
display(Data["home_ownership"].value_counts(dropna=False))

Data["home_ownership"] = Data["home_ownership"].replace("NONE", "OTHER")

print("After combining NONE into OTHER:")
display(Data["home_ownership"].value_counts(dropna=False))


In [ ]:
missing_delinq = Data["delinq_2yrs"].isna().sum()
Data["delinq_2yrs"] = Data["delinq_2yrs"].fillna(0)

print(f"delinq_2yrs missing values filled with 0: {missing_delinq}")


## 7. Remove leakage, identifiers, and low-value fields

The following fields are dropped because they are IDs/links, post-outcome payment information, sparse history fields, or redundant variables that can leak information unavailable at loan application time.


In [ ]:
drop_reasons = {
    "loan_amnt": "Redundant with funded_amnt for this workflow.",
    "funded_amnt_inv": "Investor-funded amount is redundant after funding amount is retained.",
    "pymnt_plan": "Too skewed to add useful signal.",
    "issue_d": "Origination date is not used in the baseline modeling setup.",
    "url": "Record URL/identifier, not a borrower feature.",
    "mths_since_last_record": "Very sparse credit-history field.",
    "out_prncp": "Post-origination outstanding principal; leaks repayment status.",
    "out_prncp_inv": "Post-origination outstanding principal; leaks repayment status.",
    "total_pymnt": "Post-outcome payment amount; leaks repayment behavior.",
    "total_pymnt_inv": "Post-outcome payment amount; leaks repayment behavior.",
    "total_rec_prncp": "Post-outcome received principal; leaks repayment behavior.",
    "total_rec_int": "Post-outcome received interest; leaks repayment behavior.",
    "total_rec_late_fee": "Post-outcome late fees; leaks repayment behavior.",
    "recoveries": "Post-chargeoff recovery amount; directly tied to outcome.",
    "collection_recovery_fee": "Post-chargeoff recovery fee; directly tied to outcome.",
    "last_pymnt_d": "Post-origination payment date; leaks repayment behavior.",
    "last_pymnt_amnt": "Post-origination payment amount; leaks repayment behavior.",
    "next_pymnt_d": "Future payment field, not available at application time.",
    "mths_since_last_delinq": "Sparse delinquency-history field.",
    "last_credit_pull_d": "Post-application credit-pull date.",
    "last_fico_range_high": "Latest FICO can include post-application information.",
    "last_fico_range_low": "Latest FICO can include post-application information.",
    "earliest_cr_line": "Dropped in this version to keep the first dataset compact.",
}

drop_summary = pd.DataFrame(
    [{"feature": feature, "reason": reason} for feature, reason in drop_reasons.items()]
)
display(drop_summary)


In [ ]:
drop_cols = [col for col in drop_reasons if col in Data.columns]
missing_drop_cols = [col for col in drop_reasons if col not in Data.columns]

Data = Data.drop(columns=drop_cols)

print(f"Requested drop columns: {len(drop_reasons)}")
print(f"Columns found and dropped: {len(drop_cols)}")
print(f"Columns already absent: {len(missing_drop_cols)}")
print(f"Cleaned data shape: {Data.shape}")


## 8. Save cleaned data

Save the cleaned full dataset and a small preview sample for quick inspection.


In [ ]:
Data.to_excel(CLEAN_DATA_PATH, index=False)
Data.head(100).to_excel(SAMPLE_PATH, index=False)

print(f"Saved cleaned data to: {CLEAN_DATA_PATH}")
print(f"Saved sample preview to: {SAMPLE_PATH}")


In [ ]:
display(Data.head())
Data.info()
